## _**BRONZE TABLE CREATION**_

In [1]:
%%pyspark

orders_raw = spark.read.parquet("abfss://spark_sql_ecommerce@onelake.dfs.fabric.microsoft.com/ecomm_lakehouse.Lakehouse/Files/bronze/orders.parquet")
customers_raw = spark.read.parquet("abfss://spark_sql_ecommerce@onelake.dfs.fabric.microsoft.com/ecomm_lakehouse.Lakehouse/Files/bronze/customers.parquet")
products_raw = spark.read.parquet("abfss://spark_sql_ecommerce@onelake.dfs.fabric.microsoft.com/ecomm_lakehouse.Lakehouse/Files/bronze/products.parquet")
order_items_raw = spark.read.parquet("abfss://spark_sql_ecommerce@onelake.dfs.fabric.microsoft.com/ecomm_lakehouse.Lakehouse/Files/bronze/order_items.parquet")

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 3, Finished, Available, Finished, False)

In [2]:
%%pyspark

display(orders_raw.limit(5))

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f2bad304-ce49-4c94-a9a1-13c934ce5df1)

## _**to use spark sql we need table**_

In [3]:
%%pyspark
orders_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")
customers_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")
products_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_products")
order_items_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_order_items")

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 5, Finished, Available, Finished, False)

# _**Silver table**_

In [6]:
SELECT * FROM  bronze_orders LIMIT 4

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 8, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 9 fields>

## _**silver delta table creation**_

### silver orders

In [7]:
CREATE OR REPLACE TABLE silver_orders AS
SELECT
  CAST(OrderID AS STRING) AS OrderID,
  CAST(CustomerID AS STRING) AS CustomerID,
  TO_DATE(REPLACE(OrderDate, '/', '-')) AS OrderDate,
  TO_DATE(REPLACE(ShippedDate, '/', '-')) AS ShippedDate,
  TO_DATE(REPLACE(DeliveredDate, '/', '-')) AS DeliveredDate,
  ROUND(CAST(Price AS DOUBLE), 2) AS Price,
  INITCAP(TRIM(OrderChannel)) AS OrderChannel,
  INITCAP(TRIM(Region)) AS Region,
  TRIM(ReferredBy) AS ReferredBy
FROM bronze_orders
WHERE OrderID IS NOT NULL;

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
SELECT * FROM silver_orders LIMIT 4

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 10, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 9 fields>

### silver customer

In [9]:
CREATE OR REPLACE TABLE silver_customers AS
SELECT
  CAST(CustomerID AS STRING) AS CustomerID,
  INITCAP(TRIM(CustomerName)) AS CustomerName,
  CASE 
    WHEN LOWER(IsPremiumCustomer) IN ('yes', 'y') THEN 'Yes'
    ELSE 'No'
  END AS IsPremiumCustomer,
  TO_DATE(SignupDate) AS SignupDate
FROM bronze_customers
WHERE CustomerID IS NOT NULL;

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [10]:
SELECT * FROM silver_customers LIMIT 5

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 12, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 4 fields>

#### SILVER PRODUCTS

In [11]:
CREATE OR REPLACE TABLE silver_products AS
SELECT
  CAST(ProductID AS STRING) AS ProductID,
  INITCAP(TRIM(Category)) AS Category,
  INITCAP(TRIM(Carrier)) AS Carrier,
  ROUND(CAST(ShipmentCost AS DOUBLE), 2) AS ShipmentCost,
  CASE 
    WHEN LOWER(IsDiscounted) IN ('yes', 'y') THEN 'Yes'
    ELSE 'No'
  END AS IsDiscounted
FROM bronze_products
WHERE ProductID IS NOT NULL;

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
SELECT * FROM silver_products LIMIT 4

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 14, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 5 fields>

### SILVER ORDERS 

In [13]:
CREATE OR REPLACE TABLE silver_order_items AS
SELECT
  CAST(OrderID AS STRING) AS OrderID,
  CAST(ProductID AS STRING) AS ProductID,
  CAST(Quantity AS INT) AS Quantity
FROM bronze_order_items
WHERE Quantity IS NOT NULL AND Quantity > 0;

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [14]:
SELECT * FROM silver_order_items LIMIT 4

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 16, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 3 fields>

# _**Gold**_

In [15]:
CREATE OR REPLACE TEMP VIEW orders_enriched AS
SELECT
  o.OrderID,
  o.CustomerID,
  o.OrderDate,
  o.ShippedDate,
  o.DeliveredDate,
  o.Price,
  o.Region,
  o.OrderChannel,
  o.ReferredBy,
  c.CustomerName,
  c.IsPremiumCustomer,
  c.SignupDate,
  i.ProductID,
  i.Quantity,
  p.Category,
  p.Carrier,
  p.ShipmentCost,
  p.IsDiscounted,
  DATEDIFF(o.DeliveredDate, o.ShippedDate) AS DeliveryDelay
FROM silver_orders o
LEFT JOIN silver_customers c ON o.CustomerID = c.CustomerID
LEFT JOIN silver_order_items i ON o.OrderID = i.OrderID
LEFT JOIN silver_products p ON i.ProductID = p.ProductID;

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [16]:
select * from orders_enriched

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 18, Finished, Available, Finished, False)

<Spark SQL result set with 37 rows and 19 fields>

In [17]:
CREATE OR REPLACE TABLE gold_order_kpis AS
SELECT
  CustomerID,
  CustomerName,
  IsPremiumCustomer,
  Region,
  OrderChannel,
  COUNT(DISTINCT OrderID) AS TotalOrders,
  SUM(Quantity) AS TotalQuantity,
  SUM(Price) AS TotalRevenue,
  ROUND(AVG(DeliveryDelay), 1) AS AvgDeliveryDelayDays,
  COUNT(DISTINCT CASE WHEN DeliveryDelay > 3 THEN OrderID END) AS DelayedOrders,
  ROUND(SUM(ShipmentCost), 2) AS TotalShipmentCost,
  ROUND(SUM(CASE WHEN IsDiscounted = 'Yes' THEN Price ELSE 0 END), 2) AS RevenueFromDiscounted,
  MAX(DeliveryDelay) AS MaxDeliveryDelay
FROM orders_enriched
GROUP BY 
  CustomerID, CustomerName, IsPremiumCustomer, Region, OrderChannel;


StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
select * from gold_order_kpis

StatementMeta(, 12566f00-ebf9-432a-8612-e7a308938515, 20, Finished, Available, Finished, False)

<Spark SQL result set with 25 rows and 13 fields>